## Transform Payments Data
1. Extract Date and Time from payment_timestamp and create new columns payment_date and payment_time
1. Map payment_status to contain descriptive values  
   (1-Success, 2-Pending, 3-Cancelled, 4-Failed)
1. Write transformed data to the Silver schema 

In [0]:
df = spark.table('gizmobox.bronze.py_payments')
display(df)

payment_id,order_id,payment_timestamp,payment_status,payment_method
34,12,2024-12-06T22:30:12Z,1,PayPal
35,13,2024-12-17T05:25:34Z,2,PayPal
36,16,2024-12-09T10:45:56Z,4,Bank Transfer
37,17,2024-12-02T13:50:20Z,1,Bank Transfer
38,24,2024-12-22T08:30:55Z,1,PayPal
39,27,2024-12-03T11:45:30Z,2,Credit Card
40,28,2024-12-27T05:40:10Z,4,Credit Card
41,32,2024-12-29T09:20:40Z,1,PayPal
42,37,2024-12-23T12:10:05Z,2,Credit Card
43,38,2024-12-26T20:15:15Z,1,Credit Card


### 1. Extract Date and Time from payment_timestamp

In [0]:
from pyspark.sql import functions as F

df_extracted_payments = (
                            df.select(
                                "order_id"
                                ,"payment_id"
                                ,"payment_method"
                                ,"payment_status"
                                ,F.date_format("payment_timestamp", "yyyy-MM-dd").cast('date').alias("payment_date")
                                ,F.date_format("payment_timestamp", "HH-mm-ss").alias("payment_time")
                                )
                            )
display(df_extracted_payments)


order_id,payment_id,payment_method,payment_status,payment_date,payment_time
12,34,PayPal,1,2024-12-06,22-30-12
13,35,PayPal,2,2024-12-17,05-25-34
16,36,Bank Transfer,4,2024-12-09,10-45-56
17,37,Bank Transfer,1,2024-12-02,13-50-20
24,38,PayPal,1,2024-12-22,08-30-55
27,39,Credit Card,2,2024-12-03,11-45-30
28,40,Credit Card,4,2024-12-27,05-40-10
32,41,PayPal,1,2024-12-29,09-20-40
37,42,Credit Card,2,2024-12-23,12-10-05
38,43,Credit Card,1,2024-12-26,20-15-15


In [0]:
df_extracted_payments.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- payment_id: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: integer (nullable = true)
 |-- payment_date: date (nullable = true)
 |-- payment_time: string (nullable = true)



### 2. Map payment_status to contain descriptive values  
   (1-Success, 2-Pending, 3-Cancelled, 4-Failed)

In [0]:
df_mapped_payments = (
                            df_extracted_payments.select(
                                "order_id"
                                ,"payment_id"
                                ,"payment_method"
                                ,F.col("payment_status").alias("payment_status_code")
                                ,F.when(df_extracted_payments.payment_status == 1, 'Success')
                                   .when(df_extracted_payments.payment_status == 2, 'Pending')
                                   .when(df_extracted_payments.payment_status == 3, 'Cancelled')
                                   .when(df_extracted_payments.payment_status == 4, 'Failed')
                                   .alias("payment_status")
                                ,"payment_date"
                                ,"payment_time"
                                )
                        )
display(df_mapped_payments)

order_id,payment_id,payment_method,payment_status_code,payment_status,payment_date,payment_time
12,34,PayPal,1,Success,2024-12-06,22-30-12
13,35,PayPal,2,Pending,2024-12-17,05-25-34
16,36,Bank Transfer,4,Failed,2024-12-09,10-45-56
17,37,Bank Transfer,1,Success,2024-12-02,13-50-20
24,38,PayPal,1,Success,2024-12-22,08-30-55
27,39,Credit Card,2,Pending,2024-12-03,11-45-30
28,40,Credit Card,4,Failed,2024-12-27,05-40-10
32,41,PayPal,1,Success,2024-12-29,09-20-40
37,42,Credit Card,2,Pending,2024-12-23,12-10-05
38,43,Credit Card,1,Success,2024-12-26,20-15-15


### 3. Write transformed data to the Silver schema 

In [0]:
df_mapped_payments.writeTo('gizmobox.silver.py_payments').createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.silver.py_payments LIMIT 5;

order_id,payment_id,payment_method,payment_status_code,payment_status,payment_date,payment_time
6,1,Credit Card,1,Success,2024-10-30,04-47-27
10,2,Credit Card,2,Pending,2024-10-09,22-09-27
11,3,Bank Transfer,4,Failed,2024-10-15,17-34-19
15,4,Bank Transfer,1,Success,2024-10-22,01-47-25
19,5,PayPal,2,Pending,2024-10-15,12-40-26
